In [25]:
import pandas as pd
import numpy as np

# Fijar semilla para reproducibilidad de los datos ficticios
np.random.seed(42)

print("Generando catálogo de productos (Excel con 50 productos)...")
# 1. Crear catálogo de productos (50 productos diferentes)
ids_productos = [f"PROD_{str(i).zfill(3)}" for i in range(1, 51)]
nombres = [f"Producto Gourmet {i}" for i in range(1, 51)]
precios = np.round(np.random.uniform(500, 15000, 50), 2)

df_productos = pd.DataFrame({
    'id_producto': ids_productos,
    'nombre_producto': nombres,
    'precio_unitario': precios
})

# Inyectar algunos errores intencionales en el catálogo (nulos en precio)
df_productos.loc[5, 'precio_unitario'] = np.nan
df_productos.loc[12, 'precio_unitario'] = None

# Guardar como Excel
df_productos.to_excel('productos.xlsx', index=False, engine='openpyxl')


print("Generando transacciones (CSV con 500 registros)...")
# 2. Crear transacciones (500 registros)
# Mezclamos IDs normales y algunos con errores de formato (ej. enteros en lugar de strings, o espacios)
ids_aleatorios = np.random.choice(ids_productos, 500)
cantidades = np.random.randint(1, 10, 500).astype(float)

# Generar fechas con algunos formatos mezclados o nulos
fechas_base = pd.date_range(start='2026-01-01', periods=100, freq='D')
fechas_aleatorias = np.random.choice(fechas_base, 500)
fechas_str = [str(pd.Timestamp(f).date()) if np.random.rand() > 0.05 else "FECHA_INVALIDA" for f in fechas_aleatorias]

df_transacciones = pd.DataFrame({
    'id_producto': ids_aleatorios,
    'cantidad': cantidades,
    'fecha': fechas_str
})

# Inyectar errores intencionales en transacciones
df_transacciones.loc[10, 'cantidad'] = np.nan       # Valor nulo en cantidad
df_transacciones.loc[25, 'id_producto'] = np.nan    # Valor nulo en llave extranjera
df_transacciones.loc[50, 'fecha'] = None            # Valor nulo en fecha
# Simulamos un producto huérfano que no existe en el catálogo para probar el inner join
df_transacciones.loc[100, 'id_producto'] = 'PROD_999'

# Guardar como CSV
df_transacciones.to_csv('transacciones.csv', index=False)

print("¡Archivos 'productos.xlsx' y 'transacciones.csv' creados con éxito en Colab con errores simulados!")

Generando catálogo de productos (Excel con 50 productos)...
Generando transacciones (CSV con 500 registros)...
¡Archivos 'productos.xlsx' y 'transacciones.csv' creados con éxito en Colab con errores simulados!


In [26]:
df_transacciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_producto  499 non-null    object 
 1   cantidad     499 non-null    float64
 2   fecha        499 non-null    object 
dtypes: float64(1), object(2)
memory usage: 11.8+ KB


In [27]:
df_productos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_producto      50 non-null     object 
 1   nombre_producto  50 non-null     object 
 2   precio_unitario  48 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.3+ KB


In [28]:
print("Nulos por columna (transacciones):")
print(df_transacciones.isnull().sum())
print()
print("dtype id_producto en transacciones:", df_transacciones["id_producto"].dtype)
print("dtype id_producto en productos:   ", df_productos["id_producto"].dtype)


Nulos por columna (transacciones):
id_producto    1
cantidad       1
fecha          1
dtype: int64

dtype id_producto en transacciones: object
dtype id_producto en productos:    object


In [29]:
duplicados = df_transacciones.duplicated().sum()
transacciones = df_transacciones.drop_duplicates()
print(f"Filas duplicadas eliminadas: {duplicados}")

Filas duplicadas eliminadas: 2


Debemos mantener la columna id_producto como tipo texto (string) para que no se pierdan los datos. Procederé a corregir y volver a ejecutar la celda de procesamiento de datos (yPyMB8LN79Bi) con esta lógica corregida.

In [30]:
# Calculamos el total de la venta con los datos reales recuperados
df_final["total_venta"] = df_final["cantidad"] * df_final["precio_unitario"]

# Mostramos las primeras columnas del resultado final
df_final[["id_producto", "nombre_producto", "cantidad", "precio_unitario", "total_venta"]].head()

,id_producto,nombre_producto,cantidad,precio_unitario,total_venta
0,PROD_018,Producto Gourmet 18,9.0,8108.97,72980.73
1,PROD_026,Producto Gourmet 26,5.0,11885.05,59425.25
2,PROD_044,Producto Gourmet 44,9.0,13685.15,123166.35
3,PROD_034,Producto Gourmet 34,4.0,14258.84,57035.36
4,PROD_010,Producto Gourmet 10,5.0,10767.05,53835.25


In [31]:
# Realizamos un merge temporal de tipo 'left' con indicador para detectar huérfanos
merge_diagnostico = pd.merge(df_transacciones, df_productos, on='id_producto', how='left', indicator=True)

# Filtramos aquellas transacciones cuyo producto no existe en el catálogo
huerfanas = merge_diagnostico[merge_diagnostico["_merge"] == "left_only"]

print(f"Transacciones con producto no encontrado en el catálogo: {len(huerfanas)}")
huerfanas[["id_producto", "cantidad", "fecha"]]

Transacciones con producto no encontrado en el catálogo: 2


,id_producto,cantidad,fecha
25,NaN,4.0,FECHA_INVALIDA
100,PROD_999,5.0,2026-03-08


In [32]:
# Exportación a Parquet
df_final.to_parquet('ventas_procesadas.parquet', index=False)
print(f"¡Transformación completada con éxito! Filas resultantes: {len(df_final)}")

¡Transformación completada con éxito! Filas resultantes: 460


```markdown
# Generación del archivo README.md

Ejecuta la siguiente celda para crear un archivo de documentación estructurado (`README.md`) en tu directorio actual. Este archivo servirá para explicar el funcionamiento de todo el pipeline de datos.
```

In [33]:
readme_content = """# Pipeline de Integración de Ventas - Gourmet Data

Este proyecto implementa un pipeline de datos (ETL) básico en Python utilizando **Pandas** para procesar, limpiar y unificar la información de transacciones de ventas con el catálogo de productos.

## 📁 Estructura del Proyecto

*   `productos.xlsx`: Catálogo de productos que contiene el identificador único (`id_producto`), nombre y precio unitario.
*   `transacciones.csv`: Registro diario de ventas que incluye el ID del producto vendido, la cantidad y la fecha.
*   `ventas_procesadas.parquet`: Dataset final limpio, consolidado y optimizado en formato columnar Parquet.

## 🛠️ Reglas de Limpieza y Transformación (ETL)

El pipeline realiza las siguientes tareas críticas de calidad de datos:

1.  **Normalización de Llaves Alfanuméricas**:
    *   Mantiene los identificadores `id_producto` como tipo texto (evitando truncar prefijos como `PROD_`).
    *   Elimina espacios en blanco innecesarios.
    *   Descarta valores nulos o cadenas vacías en esta clave.
2.  **Limpieza de Registros Incompletos**:
    *   Se eliminan transacciones con cantidades nulas o formatos de fecha inválidos (ej. "FECHA_INVALIDA").
    *   Se eliminan productos del catálogo que no tengan un precio unitario definido.
3.  **Cruce de Datos (Merge)**:
    *   Se aplica un *Inner Join* basado en `id_producto` para asegurar que solo se procesen transacciones de productos existentes en el catálogo oficial (excluyendo registros huérfanos como `PROD_999`).
4.  **Cálculo de Métricas**:
    *   Creación de la columna calculada `total_venta` (`cantidad * precio_unitario`).
5.  **Exportación Optimizada**:
    *   Guardado del set de datos final en formato `.parquet` para un almacenamiento eficiente y consultas rápidas.

## 🚀 Cómo ejecutar el pipeline

1. Asegúrate de tener instaladas las dependencias necesarias:
   ```bash
   pip install pandas openpyxl pyarrow
   ```
2. Ejecuta las celdas de procesamiento de datos en el Notebook de Colab para procesar las fuentes y generar el archivo exportado.
"""

# Escribir el contenido en el archivo README.md
with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("¡Archivo 'README.md' creado con éxito en tu entorno de trabajo!")

¡Archivo 'README.md' creado con éxito en tu entorno de trabajo!
